# Photos with no snake in them

Our detector and segmenter learned only from snake photos. At the Milestone 3 review the detector drew a snake
box round a person and round a lion printed on a T-shirt, and we had never measured how often it sees a snake
where there is none. This notebook collects about 3,000 openly licensed photos that hold no snake:

| Source | What we take | Photos |
|---|---|---|
| Open Images V7 (CC BY 2.0) | people, hands and arms, T-shirts, dogs, cats, other animals, ropes, hoses and belts | 2,500 |
| iNaturalist (CC0 or CC BY) | snake lookalikes: glass lizards, caecilians, eels, earthworms | 500 |

The photos are split 60/20/20 into train, val and test, with each photographer's photos kept in one split.
Later notebooks train the detector and segmenter with the train split, pick the "is there a snake?" rule on
val, and report the false-snake rate once on test.

Every photo's licence, source page and owner are in `manifest.csv`. No photo goes in the repo; this notebook's
Kaggle output (`images/` and `manifest.csv`) is the set, read later as kernel `puneetbajajiitm/sg-m4-non-snake`.
It runs on Kaggle on a CPU with internet on, in about ten minutes.

In [1]:
import html
import json
import math
import sys
from io import BytesIO
from pathlib import Path

import pandas as pd
from PIL import Image

sys.path.insert(0, str(Path("../src").resolve()))
from serpentguard import negatives

LIMIT = None  # a number makes a smoke run with about that many photos
SEED = 42
ON_KAGGLE = Path("/kaggle/working").exists()
OUT = Path("/kaggle/working") if ON_KAGGLE else Path("../../kaggle/m4_runs/non_snake/local_out")
TABLES = Path("/tmp/open_images_tables") if ON_KAGGLE else OUT / "tables"  # Open Images label files, not saved
OUT.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

## 1. Which Open Images labels mean "snake"

Open Images tags each photo with labels that people checked: confidence 1 means the thing is in the photo, 0
means it was checked and is not. Snake is `/m/078jl`. The 600-class hierarchy that ships with Open Images is
where child classes of Snake would be listed. The full list of 20,000 labels also has species-level snake classes
(King cobra, Garter snake and so on), so every label whose name names a snake counts as a snake label too. That
also catches a few things that are not snakes, such as cars named Cobra and feather boas; dropping those photos
costs nothing.

In [2]:
OI_TABLES = "https://storage.googleapis.com/openimages"
OI_IMAGES = "https://open-images-dataset.s3.amazonaws.com"
OI_SPLITS = {"validation": "val", "test": "test"}  # the two small splits; their labels were checked by people
OI_LICENCE = "https://creativecommons.org/licenses/by/2.0/"


def table(path):
    local = TABLES / Path(path).name
    if not local.exists():
        local.write_bytes(negatives.get(f"{OI_TABLES}/{path}").content)
    return local


classes = pd.read_csv(table("v7/oidv7-class-descriptions.csv"))
hierarchy = json.loads(table("2018_04/bbox_labels_600_hierarchy.json").read_text())
names = classes.set_index("LabelName")["DisplayName"]
snake_mids = negatives.snake_mids(classes, hierarchy)
children = negatives.snake_mids(classes[classes["LabelName"] == negatives.OI_SNAKE], hierarchy) - {negatives.OI_SNAKE}
print(f"{negatives.OI_SNAKE} is '{names[negatives.OI_SNAKE]}' in the class list; "
      f"classes under it in the hierarchy: {sorted(children) or 'none'}")
print(f"{len(snake_mids)} snake labels in all:")
print(", ".join(sorted(names[sorted(snake_mids)])))

/m/078jl is 'Snake' in the class list; classes under it in the hierarchy: none
102 snake labels in all:
Ac cobra, Adder's tongue, Banded water snake, Black mamba, Boa, Boa constrictor, Boas, Brown snake, Bullsnake, Burmese python, Checkered Garter Snake, Chrysler viper gts-r, Cobra, Colubridae, Common European viper, Common Garter Snake, Common Kingsnake, Coral snake, Corn Snake, Cottonmouth, Diamondback water snake, Dodge Viper, Eastern Hognose Snake, Eastern Indigo Snake, Eastern brown snake, Eastern coral snake, Eastern diamondback rattlesnake, Eastern racer, Elapidae, Feather boa, Flying snake, Ford falcon cobra, Ford shelby cobra concept, Garter, Garter snake, Glossy snake, Gopher snake (Species), Grass snake, Green snakes, Ground rattlesnake, Groundsnake, Hennessey viper venom 1000 twin turbo, Hog-nosed snakes, Hognose snake, Horned viper, Indian cobra, King cobra, Kingsnake, Lake Erie Water Snake, Long nosed Snake, Lyre snake, Mamba, Massasauga, Milk snake, Moccasin, Mojave Ratt

## 2. The photos we want from Open Images

We use the validation and test splits only (167,056 photos): every label there was checked by people, and their
label files are small. A photo is out if any snake label is on it: a label people checked, a machine-made label
at any confidence, or a snake box. Each photo is taken once. Rarer categories fill first, so a photo that has
both a T-shirt and a person counts as a T-shirt. Hose and Garden hose exist as labels, but only one checked photo
carries them (the same photo carries both), so rope and belt fill the rest of that category's 300.

In [3]:
TARGETS = {  # category: {Open Images label: photos}, filled in this order
    "rope_hose_belt": {"Hose": 1, "Garden hose": 1, "Rope": 150, "Belt": 149},
    "tshirt": {"T-shirt": 300},
    "other_animal": {name: 30 for name in ("Lion", "Tiger", "Leopard", "Horse", "Cattle", "Monkey", "Bird",
                                           "Lizard", "Frog", "Crocodile")},
    "cat": {"Cat": 200},
    "dog": {"Dog": 200},
    "hand_arm": {"Human hand": 150, "Human arm": 150},
    "person": {"Person": 900},
}
mid_of = {}
for name in [name for labels in TARGETS.values() for name in labels]:
    mids = classes.loc[classes["DisplayName"] == name, "LabelName"]
    assert len(mids) == 1, f"{name} must name exactly one label"
    mid_of[name] = mids.iloc[0]
category_of = {mid_of[name]: category for category, labels in TARGETS.items() for name in labels}
wanted = {mid_of[name]: n for labels in TARGETS.values() for name, n in labels.items()}

human = pd.concat(pd.read_csv(table(f"v7/oidv7-{short}-annotations-human-imagelabels.csv"))
                  for short in OI_SPLITS.values())
machine = pd.concat(pd.read_csv(table(f"v5/{split}-annotations-machine-imagelabels.csv")) for split in OI_SPLITS)
boxes = pd.concat(pd.read_csv(table(f"v5/{split}-annotations-bbox.csv"), usecols=["ImageID", "LabelName"])
                  for split in OI_SPLITS)
snake_sources = {"checked by people": human[human["LabelName"].isin(snake_mids) & (human["Confidence"] == 1)],
                 "machine-made": machine[machine["LabelName"].isin(snake_mids)],
                 "snake box": boxes[boxes["LabelName"].isin(snake_mids)].assign(Confidence=1.0)}
for source, rows in snake_sources.items():
    print(f"photos with a snake label {source}: {rows['ImageID'].nunique():,}")
snake_ids = set().union(*(set(rows["ImageID"]) for rows in snake_sources.values()))
print(f"photos out because of a snake label: {len(snake_ids):,} of {human['ImageID'].nunique():,}")

columns = ["ImageID", "LabelName", "Confidence"]
labels = pd.concat([human[columns]] + [rows[columns] for rows in snake_sources.values()], ignore_index=True)
picked = negatives.open_images_candidates(labels, wanted, snake_mids, seed=SEED)
checked = labels[(labels["Confidence"] == 1) & ~labels["ImageID"].isin(snake_ids)]
available = checked.groupby("LabelName")["ImageID"].nunique()
pd.DataFrame({"category": [category_of[m] for m in wanted], "label": [names[m] for m in wanted],
              "wanted": list(wanted.values()), "photos with the label": [available.get(m, 0) for m in wanted],
              "picked": [int((picked["LabelName"] == m).sum()) for m in wanted]})

photos with a snake label checked by people: 479
photos with a snake label machine-made: 648
photos with a snake label snake box: 295
photos out because of a snake label: 688 of 167,056


,category,label,wanted,photos with the label,picked
0,rope_hose_belt,Hose,1,1,1
1,rope_hose_belt,Garden hose,1,1,0
2,rope_hose_belt,Rope,150,537,150
3,rope_hose_belt,Belt,149,354,149
4,tshirt,T-shirt,300,781,300
5,other_animal,Lion,30,202,30
6,other_animal,Tiger,30,90,30
7,other_animal,Leopard,30,160,30
8,other_animal,Horse,30,1114,30
9,other_animal,Cattle,30,539,30


**Negative control.** We plant a made-up photo carrying both Person and Snake among every photo with a Person
label, and ask for all of them. The planted photo, and every real photo with a snake label, must be missing,
while every other Person photo is taken. Asking for all of them shows it is the filter, not the random draw,
that leaves them out.

In [4]:
PLANTED = "planted_person_and_snake"
person = mid_of["Person"]
person_ids = set(labels.loc[(labels["LabelName"] == person) & (labels["Confidence"] == 1), "ImageID"])
planted = pd.DataFrame({"ImageID": [PLANTED, PLANTED], "LabelName": [person, negatives.OI_SNAKE],
                        "Confidence": [1.0, 1.0]})
everyone = pd.concat([labels[labels["ImageID"].isin(person_ids)], planted])
taken = set(negatives.open_images_candidates(everyone, {person: len(person_ids) + 1}, snake_mids)["ImageID"])
assert PLANTED not in taken and not taken & snake_ids
assert taken == person_ids - snake_ids
print(f"Person photos: {len(person_ids):,}; with a snake label: {len(person_ids & snake_ids)}; "
      f"taken when asking for all: {len(taken):,}; planted photo taken: {PLANTED in taken}")

Person photos: 32,573; with a snake label: 152; taken when asking for all: 32,421; planted photo taken: False


Each picked photo's licence, Flickr page and author come from the Open Images image lists. We keep only photos
listed as CC BY 2.0, and check that the image path in the public bucket works for one photo of each split.

In [5]:
info = pd.concat(pd.read_csv(table(f"2018_04/{split}/{split}-images-with-rotation.csv"),
                             usecols=["ImageID", "Subset", "OriginalLandingURL", "License", "Author"])
                 for split in OI_SPLITS).set_index("ImageID")
oi = picked.join(info, on="ImageID")
print(f"licences of the picked photos: {oi['License'].value_counts(dropna=False).to_dict()}")
oi = oi[oi["License"] == OI_LICENCE].assign(Author=lambda rows: rows["Author"].map(html.unescape))  # "O&#x27;Connor"
oi_rows = pd.DataFrame({
    "source": "openimages", "source_id": oi["ImageID"], "owner": oi["Author"], "licence": "cc-by-2.0",
    "url": oi["OriginalLandingURL"], "category": oi["LabelName"].map(category_of), "label": oi["LabelName"].map(names),
    "image_url": OI_IMAGES + "/" + oi["Subset"] + "/" + oi["ImageID"] + ".jpg",
    "attribution": "(c) " + oi["Author"] + ", CC BY 2.0"})
for split in OI_SPLITS:
    first = oi_rows[oi["Subset"] == split].iloc[0]
    with Image.open(BytesIO(negatives.get(first["image_url"]).content)) as image:
        print(f"{split}: {first['image_url']} -> {image.size[0]}x{image.size[1]} {image.format}")

licences of the picked photos: {'https://creativecommons.org/licenses/by/2.0/': 2500}
validation: https://open-images-dataset.s3.amazonaws.com/validation/b02168c1931d5ad8.jpg -> 683x1024 JPEG
test: https://open-images-dataset.s3.amazonaws.com/test/5948cd151bb8be9c.jpg -> 1024x680 JPEG


## 3. Snake lookalikes from iNaturalist

Animals people mistake for snakes. We query iNaturalist by taxon id, never by name, because a name search once
returned a damselfly. Each id's record is fetched first and shown below, and `inat_lookalikes` stops if a record's
name is not the one we expect. We take research-grade observations with a CC0 or CC BY photo, one photo per
observation, with photographers taking turns. An observation whose taxon sits under Snakes (suborder Serpentes,
id 85553), or outside the queried taxon, is dropped.

In [6]:
LOOKALIKES = {  # scientific name: iNaturalist taxon id
    "Anguinae": 797527,          # glass lizards, legless lizards found in north-east India too
    "Gymnophiona": 27880,        # caecilians
    "Anguilliformes": 47314,     # true eels
    "Synbranchiformes": 53512,   # swamp and spiny eels, common in Indian paddy fields
    "Crassiclitellata": 152943,  # earthworms
}
CATEGORY_OF_TAXON = {"Anguinae": "legless_lizard", "Gymnophiona": "caecilian", "Anguilliformes": "eel",
                     "Synbranchiformes": "eel", "Crassiclitellata": "earthworm"}
PER_TAXON = 100

In [7]:
records = pd.DataFrame([negatives.inat_taxon(negatives.SERPENTES_ID)]
                       + [negatives.inat_taxon(taxon_id) for taxon_id in LOOKALIKES.values()])
assert records.iloc[0]["name"] == "Serpentes" and list(records["name"][1:]) == list(LOOKALIKES)
records

,id,name,rank,common_name
0,85553,Serpentes,suborder,Snakes
1,797527,Anguinae,subfamily,Glass Lizards
2,27880,Gymnophiona,order,Caecilians
3,47314,Anguilliformes,order,True Eels
4,53512,Synbranchiformes,order,Swamp Eels
5,152943,Crassiclitellata,order,Typical Earthworms


In [8]:
inat = negatives.inat_lookalikes(LOOKALIKES, PER_TAXON, seed=SEED)
inat_rows = inat.assign(category=inat["taxon"].map(CATEGORY_OF_TAXON), label=inat["taxon"])
print(inat_rows.groupby(["category", "taxon"]).agg(photos=("source_id", "size"), photographers=("owner", "nunique"))
      .to_string())
print(f"licences: {inat_rows['licence'].value_counts().to_dict()}")

                                 photos  photographers
category       taxon                                  
caecilian      Gymnophiona          100             81
earthworm      Crassiclitellata     100            100
eel            Anguilliformes       100            100
               Synbranchiformes     100            100
legless_lizard Anguinae             100            100
licences: {'cc-by': 412, 'cc0': 88}


## 4. Download

Photos are saved at most 500 px on the long side, the size of our snake photos, so that size alone can't tell a
snake photo from a non-snake one. Downloads go four at a time, and each request is retried with doubling waits on
HTTP 429 and 5xx. A photo that still fails is listed and left out.

In [9]:
frame = pd.concat([oi_rows, inat_rows], ignore_index=True)
if LIMIT:
    per_category = math.ceil(LIMIT / frame["category"].nunique())
    frame = frame.sample(frac=1, random_state=SEED).groupby("category").head(per_category).head(LIMIT)
frame = frame.assign(file="images/" + frame["category"] + "/" + frame["source"] + "_" + frame["source_id"] + ".jpg")
assert frame["file"].is_unique
frame = negatives.download_photos(frame, OUT, workers=4)
failed = frame[frame["error"] != ""]
print(f"saved {len(frame) - len(failed):,} of {len(frame):,} photos; failed: {len(failed)}")
if len(failed):
    print(failed[["file", "error"]].head(20).to_string())
frame = frame[frame["error"] == ""].drop(columns="error").reset_index(drop=True)

saved 3,000 of 3,000 photos; failed: 0


## 5. Near-duplicates, then the split

The same photo can appear twice (an iNaturalist user uploading one shot to two observations, or a Flickr photo
used twice). We use the perceptual hash of notebooks 03 and 09 and drop any photo within 8 bits of an earlier one.
First we check the check: shrunk, re-encoded copies of 20 photos must be caught.

In [10]:
def file_hash(path):
    with Image.open(path) as image:
        return negatives.phash(image)


frame["phash"] = [file_hash(OUT / file) for file in frame["file"]]
hash_control = frame.sample(min(20, len(frame)), random_state=SEED)
caught = 0
for row in hash_control.itertuples():
    buffer = BytesIO()
    with Image.open(OUT / row.file) as image:
        image.resize((int(image.width * 0.8), int(image.height * 0.8))).save(buffer, "JPEG", quality=70)
    bits = bin(int(file_hash(buffer), 16) ^ int(row.phash, 16)).count("1")
    caught += bits <= negatives.NEAR_DUP_BITS
print(f"control: {caught} of {len(hash_control)} re-encoded copies caught")
assert caught == len(hash_control)

repeats = negatives.near_duplicate_rows(frame["phash"])
print(f"near-duplicates dropped: {len(repeats)}")
if repeats:
    print(frame.loc[repeats, ["file", "owner"]].head(20).to_string())
frame = frame.drop(index=repeats).reset_index(drop=True)

control: 20 of 20 re-encoded copies caught
near-duplicates dropped: 1
                                            file         owner
2692  images/caecilian/inaturalist_333359633.jpg  demetztobias


The split keeps every owner (Flickr author or iNaturalist user) in one split, and fills each category's 60/20/20
as it goes, so every split has every category.

In [11]:
frame = negatives.split_negatives(frame, seed=SEED)
counts = pd.crosstab(frame["category"], frame["split"], margins=True, margins_name="all").reindex(
    columns=[*negatives.SPLITS, "all"], fill_value=0)
print(f"owners: {frame['owner'].nunique():,}; split shares: "
      f"{frame['split'].value_counts(normalize=True).round(3).to_dict()}")
counts

owners: 2,639; split shares: {'train': 0.6, 'test': 0.2, 'val': 0.2}


split,train,val,test,all
category,,,,
caecilian,59,20,20,99
cat,120,40,40,200
dog,120,40,40,200
earthworm,60,20,20,100
eel,120,40,40,200
hand_arm,180,60,60,300
legless_lizard,60,20,20,100
other_animal,180,60,60,300
person,540,180,180,900


## 6. Checks before saving

Every row must have an allowed licence, a source page and an owner, with no empty or placeholder values. No
owner may sit in two splits, no two photos in different splits may be near-duplicates, every file must open, no
Open Images photo may carry a snake label, and no lookalike may sit under Snakes.

In [12]:
manifest = frame[negatives.MANIFEST_COLUMNS + ["label", "source_id", "image_url", "attribution", "width", "height",
                                               "phash"]]
assert set(manifest["licence"]) <= {"cc-by-2.0", "cc-by", "cc0"}, set(manifest["licence"])
assert manifest["url"].str.startswith("https://").all() and manifest["image_url"].str.startswith("https://").all()
text = manifest.astype(str)
empty = text.apply(lambda column: column.str.strip().isin(["", "nan", "None", "null", "NaN"])).sum()
assert empty.sum() == 0, empty[empty > 0]
placeholder = text.apply(lambda column: column.str.contains("error|placeholder|TODO|lorem", case=False)).sum()
print(f"cells with an error or placeholder word: {placeholder[placeholder > 0].to_dict() or 'none'}")
assert (manifest.groupby("owner")["split"].nunique() == 1).all()
assert negatives.cross_split_pairs(manifest) == []
assert not set(manifest.loc[manifest["source"] == "openimages", "source_id"]) & snake_ids
assert (manifest[["width", "height"]].max(axis=1) <= negatives.MAX_SIDE).all()
for file in manifest["file"]:
    with Image.open(OUT / file) as image:
        image.verify()
print(f"{len(manifest):,} rows: licences {manifest['licence'].value_counts().to_dict()}; no empty cells; "
      "no owner in two splits; no near-duplicates across splits; every file opens")

cells with an error or placeholder word: none
2,999 rows: licences {'cc-by-2.0': 2500, 'cc-by': 411, 'cc0': 88}; no empty cells; no owner in two splits; no near-duplicates across splits; every file opens


## 7. A contact sheet to look at

100 random photos on one sheet (`contact_sheet.jpg`, with `contact_sheet_credits.csv`) for a person to check for
hidden snakes and wrong categories. Below are the credits of its first 20 photos; every photo is CC BY 2.0, CC BY
or CC0.

The sheet is in this notebook's Kaggle output, not in the repo: no photo goes in the repo.

In [13]:
sheet_rows = manifest.sample(min(100, len(manifest)), random_state=SEED).reset_index(drop=True)
sheet = negatives.contact_sheet(sheet_rows, OUT)
sheet.save(OUT / "contact_sheet.jpg", quality=85)
credits = sheet_rows[["file", "category", "owner", "licence", "url"]].rename_axis("tile")
credits.to_csv(OUT / "contact_sheet_credits.csv")
print(credits.head(20).to_string())

                                                       file        category                      owner    licence                                                       url
tile                                                                                                                                                                       
0           images/hand_arm/openimages_bc629d595f5f71c5.jpg        hand_arm                      Slava  cc-by-2.0             https://www.flickr.com/photos/slava/202860705
1                images/cat/openimages_bef520975d1d2719.jpg             cat                      David  cc-by-2.0            https://www.flickr.com/photos/davidw/171024027
2     images/rope_hose_belt/openimages_0d2769d99c4ddbf6.jpg  rope_hose_belt           Yasuhiro Chatani  cc-by-2.0          https://www.flickr.com/photos/chatani/2377156546
3             images/person/openimages_4da4f708fce3d507.jpg          person                    Groupon  cc-by-2.0          https://www.flick

## 8. Save

In [14]:
manifest.to_csv(OUT / "manifest.csv", index=False)
report = {"photos": len(manifest), "owners": int(manifest["owner"].nunique()),
          "by_category_and_split": counts.drop(index="all").to_dict(orient="index"),
          "by_source": manifest["source"].value_counts().to_dict(),
          "failed_downloads": len(failed), "near_duplicates_dropped": len(repeats),
          "snake_labels": len(snake_mids), "photos_out_for_snake_label": len(snake_ids),
          "planted_snake_taken": PLANTED in taken, "phash_control_caught": f"{caught}/{len(hash_control)}",
          "limit": LIMIT}
(OUT / "non_snake_report.json").write_text(json.dumps(report, indent=1))
(OUT / "README.md").write_text("""# SerpentGuard non-snake set (private, course use only)

Made by notebooks/19_non_snake_set.ipynb. Photos with no snake in them, for training and judging the no-snake gate.

- images/<category>/<source>_<id>.jpg: at most 500 px on the long side
- manifest.csv: file, source, owner, licence, url, category, split, then label, source_id, image_url,
  attribution, width, height, phash. url is the photo's page (Flickr or the iNaturalist observation).
- Licences: Open Images V7 photos are CC BY 2.0; iNaturalist photos are CC0 or CC BY. Credit the owner.
- split: train / val / test, each owner in one split. Test is scored once, at the end.
""")
print(json.dumps(report, indent=1))

{
 "photos": 2999,
 "owners": 2639,
 "by_category_and_split": {
  "caecilian": {
   "train": 59,
   "val": 20,
   "test": 20,
   "all": 99
  },
  "cat": {
   "train": 120,
   "val": 40,
   "test": 40,
   "all": 200
  },
  "dog": {
   "train": 120,
   "val": 40,
   "test": 40,
   "all": 200
  },
  "earthworm": {
   "train": 60,
   "val": 20,
   "test": 20,
   "all": 100
  },
  "eel": {
   "train": 120,
   "val": 40,
   "test": 40,
   "all": 200
  },
  "hand_arm": {
   "train": 180,
   "val": 60,
   "test": 60,
   "all": 300
  },
  "legless_lizard": {
   "train": 60,
   "val": 20,
   "test": 20,
   "all": 100
  },
  "other_animal": {
   "train": 180,
   "val": 60,
   "test": 60,
   "all": 300
  },
  "person": {
   "train": 540,
   "val": 180,
   "test": 180,
   "all": 900
  },
  "rope_hose_belt": {
   "train": 180,
   "val": 60,
   "test": 60,
   "all": 300
  },
  "tshirt": {
   "train": 180,
   "val": 60,
   "test": 60,
   "all": 300
  }
 },
 "by_source": {
  "openimages": 2500,
  "inat

## What we found

The set has 2,999 photos: 2,500 from Open Images and 499 from iNaturalist. 2,500 are CC BY 2.0, 411 CC BY and 88 CC0,
and every row has its licence, source page and owner. All 3,000 downloads worked. One caecilian photo was a near-copy
of another and was dropped, so there are 99 caecilians.

- **No snakes by label.** 102 Open Images labels name a snake, and 688 of the 167,056 validation and test photos carry
  one (checked by people, machine-made or a snake box). None of them is in the set. The planted photo with Person and
  Snake was left out, and so were the 152 real Person photos that carry a snake label. No lookalike sits under Snakes
  in the iNaturalist tree.
- **No snakes on the contact sheets.** The 100 photos of `contact_sheet.jpg` (in this notebook's Kaggle
  output) were judged for a hidden snake and a wrong category, and every glass lizard, caecilian and eel photo
  (399, on four more sheets kept outside the repo) for a hidden snake; none was found. The verdict is recorded
  here, not per photo. For the lookalike photos that look most like a snake (snake eels, swamp eels, glass
  lizards), the iNaturalist identifications were checked, and all agree on the lookalike.
- **The split.** 1,799 train, 600 val and 600 test, with every category at 60/20/20. The 2,639 owners each sit in one
  split, and no two photos in different splits are near-copies.
- **Worth knowing.** The lookalikes are hard on purpose: snake eels look like sea kraits, and glass lizards look like
  snakes. If the retrained gate starts missing real sea snakes or slender snakes, these photos are the first place to
  look. A few photos show their label only as a small part of the scene, for example a dog on a lead filed under
  rope and belt, or a heron holding an eel. Open Images has only one checked photo of a hose.
